# Face Recognition: Detection, Alignment, Embedding and Threshold Calibration

A reproducible walkthrough of a working face-identification pipeline — and of
the mistakes that quietly cost accuracy in most hobby implementations.

**Stack:** OpenCV DNN only. No PyTorch, no TensorFlow, no GPU. Runs on a laptop.

| Stage | Model | Output |
|---|---|---|
| Detection | YuNet (`face_detection_yunet_2023mar.onnx`, 227 KB) | box + 5 landmarks |
| Alignment | `SFace.alignCrop()` | canonical 112x112 |
| Embedding | SFace (`face_recognition_sface_2021dec.onnx`, 37 MB) | 128-D vector |
| Matching | `SFace.match()` | cosine / L2 |

---

## The headline result

Two photographs of one person, taken 16 months apart:

| Pipeline | cosine | verdict |
|---|---|---|
| Haar box → `cv2.resize(112,112)` *(the original build)* | **0.175** | ✗ false mismatch |
| YuNet → `SFace.alignCrop()` *(this notebook)* | **0.642** | ✓ correct match |

That is a **3.7x** improvement in the identity signal on identical inputs and an
identical recognition model — the difference is entirely preprocessing.

### Which half of the fix did the work?

Section 3 isolates it, and the answer is more interesting than the headline.
Swapping *only* the resize for `alignCrop`, while keeping YuNet's box, gives
roughly **1.1x** on these near-frontal photos — measured live in that cell, not
quoted from memory.

So most of the 3.7x came from **replacing the detector**, not from alignment
alone. Haar's box is framed differently — looser, often off-centre, frontal-only
— and feeding that to a resize compounds the error. Alignment then removes what
remains.

The lesson generalises: **alignment matters most when pose varies.** On two
cooperative frontal photos a plain resize can still clear the threshold. On
profile shots, tilted heads, or a detector that frames inconsistently, it will
not — and that is precisely when you need recognition to hold up.

Run section 3 on your own photos and read the number it prints rather than this
table: if your pair is near-frontal, expect a modest gap; if it is not, expect a
large one.


In [ ]:
import sys, json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

BACKEND = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BACKEND))

from app.services.face_processor import (
    detect_faces_detailed, align_face, encode_aligned_face, feature_to_list,
    evaluate_face_similarity, match_features, embed_primary_face,
    models_ready, SFACE_L2_THRESHOLD, SFACE_COSINE_THRESHOLD,
)

print(json.dumps(models_ready(), indent=2))

## 1. Load images

Drop your own photographs into `backend/tests/fixtures/` (gitignored):

```
same_person/       two different photos of ONE person
different_person/  that person, and someone else
```

**A negative pair is not optional.** A system evaluated only on matches will
happily accept everybody and score 100%.

In [ ]:
FIX = BACKEND / "tests" / "fixtures"

def load(folder):
    out = []
    d = FIX / folder
    if d.exists():
        for f in sorted(d.iterdir()):
            if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
                img = cv2.imdecode(np.fromfile(str(f), dtype=np.uint8), cv2.IMREAD_COLOR)
                if img is not None:
                    out.append((f.name, img))
    return out

same = load("same_person")
diff = load("different_person")
print("same_person     :", [n for n, _ in same])
print("different_person:", [n for n, _ in diff])
if len(same) < 2 or len(diff) < 2:
    print("\nAdd two images to each folder to run the rest of this notebook.")

## 2. Detection — why not Haar

The classic `haarcascade_frontalface_default` is frontal-only and brittle under
lighting change. Worse, implementations that bolt on a "skin-colour blob"
fallback will happily return an arm or a wall as a face.

YuNet is a small CNN that returns something Haar cannot: **five landmarks**
(both eyes, nose, both mouth corners). Section 3 shows why that changes
everything.

In [ ]:
if same:
    name, img = same[0]
    dets, W, H = detect_faces_detailed(img)
    print(f"{name}: {W}x{H}, {len(dets)} face(s)")

    vis = img.copy()
    for d in dets:
        b = d.box
        cv2.rectangle(vis, (b['left'], b['top']), (b['right'], b['bottom']), (120, 220, 150), 3)
        for (x, y) in d.landmarks:
            cv2.circle(vis, (x, y), max(2, W // 300), (80, 200, 255), -1)
        print(f"  score {d.score:.3f}  box {b}")
        print(f"  landmarks {d.landmarks}")

    plt.figure(figsize=(7, 7))
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title("YuNet: box + 5 landmarks"); plt.show()

## 3. Alignment — isolating its contribution

SFace is trained on faces warped so the eyes sit on a fixed horizontal line and
the nose and mouth land at fixed offsets. `alignCrop()` performs that similarity
transform from the landmarks.

A very common shortcut skips it:

```python
aligned = cv2.resize(face_box, (112, 112))   # variable named "aligned"...
feature = recognizer.feature(aligned)        # ...but never aligned
```

This **looks** fine: the model runs, returns 128 numbers, nothing errors. The
numbers just carry less identity.

The cell below is a controlled comparison — **same detector, same box, same
model**, differing only in the warp. That isolates alignment's contribution from
the detector swap, so the number it prints is the honest one. Expect it to be
modest on frontal photos and to grow with pose variation.


In [ ]:
from app.services.face_processor import _RECOGNIZER, crop_face_region

def embed_naive(img, det):
    """The shortcut: padded box, plain resize, no landmark warp."""
    crop = crop_face_region(img, det.box)
    return _RECOGNIZER.feature(cv2.resize(crop, (112, 112)))

def embed_aligned(img, det):
    """The correct path: landmark-driven alignCrop."""
    return encode_aligned_face(align_face(img, det))

if len(same) >= 2:
    (n1, i1), (n2, i2) = same[0], same[1]
    d1 = detect_faces_detailed(i1)[0][0]
    d2 = detect_faces_detailed(i2)[0][0]

    cos_naive, l2_naive = match_features(embed_naive(i1, d1), embed_naive(i2, d2))
    cos_align, l2_align = match_features(embed_aligned(i1, d1), embed_aligned(i2, d2))

    print(f"Same person: {n1}  vs  {n2}\n")
    print(f"  plain resize   cosine {cos_naive:+.4f}   L2 {l2_naive:.4f}   "
          f"{'MATCH' if l2_naive <= SFACE_L2_THRESHOLD else 'MISMATCH'}")
    print(f"  alignCrop      cosine {cos_align:+.4f}   L2 {l2_align:.4f}   "
          f"{'MATCH' if l2_align <= SFACE_L2_THRESHOLD else 'MISMATCH'}")
    if cos_naive > 0:
        print(f"\n  alignment improves the identity signal {cos_align / cos_naive:.2f}x")

    fig, ax = plt.subplots(2, 2, figsize=(7, 7))
    for col, (img, det, nm) in enumerate([(i1, d1, n1), (i2, d2, n2)]):
        ax[0, col].imshow(cv2.cvtColor(cv2.resize(crop_face_region(img, det.box), (112, 112)), cv2.COLOR_BGR2RGB))
        ax[0, col].set_title(f"plain resize\n{nm}", fontsize=9)
        ax[1, col].imshow(cv2.cvtColor(align_face(img, det), cv2.COLOR_BGR2RGB))
        ax[1, col].set_title("alignCrop", fontsize=9)
    for a in ax.ravel(): a.axis("off")
    plt.suptitle("Note how alignCrop puts the eyes in the same place every time")
    plt.tight_layout(); plt.show()

## 4. The embedding

128 floats, L2-normalised to the unit sphere. Two useful properties:

- Cosine similarity is then just a dot product.
- Cosine and L2 become **the same decision**, because for unit vectors
  `L2 = sqrt(2(1 - cos))`.

In [ ]:
if same:
    r = embed_primary_face(same[0][1])
    e = np.array(r["embedding"])
    print(f"dimension {e.shape[0]}   norm {np.linalg.norm(e):.6f}")
    print(f"first 8: {np.round(e[:8], 4)}")

    fig, ax = plt.subplots(figsize=(11, 1.6))
    ax.imshow(e.reshape(1, -1), aspect="auto", cmap="RdYlGn")
    ax.set_yticks([]); ax.set_xlabel("dimension")
    ax.set_title("128-D identity embedding")
    plt.tight_layout(); plt.show()

    lhs = SFACE_L2_THRESHOLD
    rhs = np.sqrt(2 * (1 - SFACE_COSINE_THRESHOLD))
    print(f"\npublished thresholds: cosine >= {SFACE_COSINE_THRESHOLD}, L2 <= {lhs}")
    print(f"sqrt(2*(1-{SFACE_COSINE_THRESHOLD})) = {rhs:.4f}  ->  the same boundary")

## 5. Evaluation — both directions

A genuine pair **and** an impostor pair. Reporting only the first is the second
classic mistake.

In [ ]:
results = []
if len(same) >= 2:
    a = embed_primary_face(same[0][1])["embedding"]
    b = embed_primary_face(same[1][1])["embedding"]
    m, pct, l2, cos = evaluate_face_similarity(a, b)
    results.append(("same person", same[0][0], same[1][0], cos, l2, pct, m, True))

if len(diff) >= 2:
    a = embed_primary_face(diff[0][1])["embedding"]
    b = embed_primary_face(diff[1][1])["embedding"]
    m, pct, l2, cos = evaluate_face_similarity(a, b)
    results.append(("different people", diff[0][0], diff[1][0], cos, l2, pct, m, False))

print(f"{'pair':<18}{'cosine':>9}{'L2':>9}{'sim%':>8}{'verdict':>12}{'expected':>11}  ok")
print("-" * 76)
for label, f1, f2, cos, l2, pct, m, exp in results:
    ok = "PASS" if m == exp else "FAIL"
    print(f"{label:<18}{cos:>+9.4f}{l2:>9.4f}{pct:>8.2f}"
          f"{('MATCH' if m else 'NON-MATCH'):>12}{('MATCH' if exp else 'NON-MATCH'):>11}  {ok}")

## 6. Where the threshold comes from

`1.128` is OpenCV's published operating point, not a number anyone tuned to
make a demo pass. Reference measurements from this pipeline:

| Case | L2 | similarity |
|---|---|---|
| Identical file | 0.000 | 100% |
| Same person, good photos | 0.55 – 0.95 | 64 – 88% |
| Same person, hard (age gap, pose) | 0.95 – 1.10 | 55 – 65% |
| **Decision boundary** | **1.128** | **50%** |
| Different people | 1.20 – 1.45 | 35 – 47% |

Note the practical floor: different people land near **35–47%**, not near zero.
Reaching 10% would need cosine about **-0.73** — two faces pointing nearly
opposite in embedding space, which real faces essentially never produce.

In [ ]:
from app.services.face_processor import similarity_percentage

l2s = np.linspace(0, 1.5, 200)
cs = 1 - (l2s ** 2) / 2
pct = [similarity_percentage(c) for c in cs]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(l2s, pct, color="#2f6fb0", lw=2)
ax.axvline(SFACE_L2_THRESHOLD, color="#333", ls="--", label=f"threshold {SFACE_L2_THRESHOLD}")
ax.axhline(50, color="#999", ls=":")
ax.axvspan(0.55, 0.95, alpha=0.15, color="green", label="same person (typical)")
ax.axvspan(1.20, 1.45, alpha=0.15, color="red", label="different people (typical)")
ax.set_xlabel("L2 distance"); ax.set_ylabel("reported similarity %")
ax.set_title("Threshold pinned at 50% so a boundary match doesn't read as failure")
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 7. Calibrating on your own data

The published threshold is a good default, but the best boundary depends on your
camera, lighting and photo quality. Label results **Correct** / **Not them** in
the UI and the distances accumulate in `backend/data/feedback.jsonl`.

`threshold_calibration.ipynb` sweeps them and reports the best cut-off.

> **What this is not.** This calibrates the *decision boundary*. It does **not**
> fine-tune SFace — that needs tens of thousands of labelled identities and a
> GPU, and a few dozen clicks cannot move those weights. Anyone claiming a
> handful of feedback clicks retrains a face model is overselling it.

In [ ]:
from app.services.feedback import feedback_stats, suggest_threshold
print(json.dumps(feedback_stats(), indent=2))
print(json.dumps(suggest_threshold(), indent=2))

## 8. Takeaways

1. **Detector and alignment work together.** Haar box + resize scored 0.175 on
   a genuine pair; YuNet + alignCrop scored 0.642. Isolating alignment alone on
   a YuNet box gives ~1.1x on frontal photos — its value grows with pose.
2. **Use the landmarks.** They are the reason to prefer YuNet over Haar.
3. **Evaluate both directions.** A negative pair is what catches a system that
   matches everybody.
4. **Don't tune the threshold to make a demo pass.** Calibrate it on labelled
   data, or keep the published value.
5. **Different people score ~40%, not ~5%.** Judge by the distance against the
   threshold, never by whether a percentage "feels" low.
6. **No fallbacks.** If a model is missing, raise. A pipeline that silently
   degrades to something that cannot recognise anyone is worse than one that
   stops.

---

## Reproduce

```bash
pip install opencv-python pillow numpy matplotlib
# models: https://github.com/opencv/opencv_zoo
#   face_detection_yunet_2023mar.onnx      (227 KB)
#   face_recognition_sface_2021dec.onnx    (37 MB)
```

Both models are Apache-2.0 from the OpenCV Zoo.

**Ethics.** Face recognition identifies people. Only run it on images you are
authorised to use, treat any match as an investigative lead rather than proof,
and note that accuracy varies across demographic groups — a documented property
of face recognition generally, not of this implementation. The application this
notebook accompanies requires an explicit authorisation acknowledgement before
a search will run.